>### Position-bin 크기 및 Hotspot 피처 조합 비교 실험
>
>**1. 실험 목적**
>
>기존 피처 조합 E + Hotspot×3 + Position-bin을 기준으로,
>Position-bin 크기와 Hotspot 피처 포함 여부가 암종 분류 성능에 미치는 영향을 분석한다.
>
>**2. 실험 설계**
>
>- **기준 피처:** E (아미노산 치환 쌍 + 유전자별 끊김형 피처 등)
>- **비교 조건**
>  - E + Position-bin (25, 50)
>  - E + Hotspot×3 + Position-bin (25, 50)
>- **고정 조건:** Position-bin 최소 등장 환자 수 5명, Hotspot 가중치 ×3
>- **모델:** Logistic Regression (동일 하이퍼파라미터 유지)
>- **검증:** 개발 데이터 Stratified 5-Fold CV 및 별도 Holdout
>- **평가 지표:** Macro F1-score
>
>**3. 검증 원칙**
>
>- 동일한 데이터 분할 및 모델 설정을 사용하여 피처 구성에 따른 성능 차이를 비교한다.
>- Position-bin 및 Hotspot 피처 선택은 각 Fold의 학습 데이터에서만 수행하여 데이터 누출을 방지한다.
>- 개발 CV 결과를 기준으로 후보를 비교하고, 홀드아웃 성능을 추가 확인한다.
>- 실험별 개발 OOF 예측값을 저장하여 후속 앙상블 비교에 활용한다.
>
>**4. 주요 실험 결과**
>
>| 피처 조합 | Bin 크기 | 개발 CV Macro F1 | Holdout Macro F1 |
>|---|---:|---:|---:|
>| E + Position-bin | 25 | 0.4901 | 0.4969 |
>| E + Position-bin | 50 | 0.4858 | 0.4991 |
>| E + Hotspot×3 + Position-bin | 25 | **0.4927** | **0.5072** |
>| E + Hotspot×3 + Position-bin | 50 | 0.4863 | 0.5057 |
>
>**5. 결론**
>
>- Hotspot×3를 포함한 조합이 포함하지 않은 조합보다 개발 CV 및 Holdout에서 높은 성능을 보였다.
>- Hotspot×3 포함 시 Bin 25가 Bin 50보다 개발 CV와 Holdout에서 모두 높은 성능을 기록했다.
>- 다만 성능 차이가 크지 않으므로 최종 채택 여부는 앙상블 성능을 기준으로 판단한다.

# E + Hotspot×3 + Position-bin 관련 실험

## 1. 실험 관련 준비

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 프로젝트 루트
PROJECT_ROOT = Path.cwd().resolve().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.features import (
    mutation_count,
    mutation_type_counts,
    amino_acid_pair_counts,
    gene_truncation_flags,
)
from src.validation import get_folds, run_cv

DATA_DIR = PROJECT_ROOT / "data"

train, test = load_data(DATA_DIR)
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)

print("X:", X.shape)
print("클래스 수:", len(le.classes_))
print("Fold 수:", len(folds))

X: (6201, 4384)
클래스 수: 26
Fold 수: 5


In [2]:
import re

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None

examples = [
    "V600E",
    "R132H",
    "WQ288fs",
    "-287fs",
    "R213*",
    "WT",
    "V600E V600E",
]

for variant in examples:
    tokens = variant.split()
    positions = [extract_position(token) for token in tokens]
    print(f"{variant:15} → {positions}")

V600E           → [600]
R132H           → [132]
WQ288fs         → [288]
-287fs          → [287]
R213*           → [213]
WT              → [None]
V600E V600E     → [600, 600]


In [3]:
from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_position(variant)

                if pos is not None:
                    feature = f"{gene}_bin_{pos // bin_size}"
                    patient_bins.add(feature)

        for feature in patient_bins:
            rows.append(row_idx)
            cols.append(feature)
            gene_bins.add(feature)

    return rows, cols, gene_bins


rows, cols, gene_bins = extract_position_bins(X)

print("Position-bin 등장 횟수:", len(rows))
print("Position-bin 후보 수:", len(gene_bins))
print("예시:", sorted(gene_bins)[:10])

Position-bin 등장 횟수: 245623
Position-bin 후보 수: 49116
예시: ['A2M_bin_0', 'A2M_bin_1', 'A2M_bin_10', 'A2M_bin_11', 'A2M_bin_12', 'A2M_bin_13', 'A2M_bin_14', 'A2M_bin_15', 'A2M_bin_16', 'A2M_bin_17']


In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from collections import Counter

class PositionBinTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        counts = Counter(cols)

        self.selected_bins_ = sorted(
            feature
            for feature, count in counts.items()
            if count >= self.min_patients
        )

        self.bin_to_idx_ = {
            feature: idx
            for idx, feature in enumerate(self.selected_bins_)
        }

        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        selected_rows = []
        selected_cols = []

        for row_idx, feature in zip(rows, cols):
            col_idx = self.bin_to_idx_.get(feature)

            if col_idx is not None:
                selected_rows.append(row_idx)
                selected_cols.append(col_idx)

        data = np.ones(len(selected_rows), dtype=np.float32)

        return csr_matrix(
            (data, (selected_rows, selected_cols)),
            shape=(len(X), len(self.selected_bins_)),
            dtype=np.float32
        )

In [5]:
# Position-bin 크기별 피처 수 비교
for bin_size in [25, 50, 100]:
    transformer = PositionBinTransformer(
        bin_size=bin_size,
        min_patients=5
    )

    transformer.fit(X)

    print(
        f"bin_size={bin_size:3d} | "
        f"선택된 피처 수={len(transformer.selected_bins_):,}"
    )

bin_size= 25 | 선택된 피처 수=13,446
bin_size= 50 | 선택된 피처 수=23,329
bin_size=100 | 선택된 피처 수=21,844


In [6]:
# 전체 후보 수와 선택된 피처 수 비교

for bin_size in [25, 50, 100]:
    rows, cols, gene_bins = extract_position_bins(
        X, bin_size=bin_size
    )

    transformer = PositionBinTransformer(
        bin_size=bin_size,
        min_patients=5
    )
    transformer.fit(X)

    total = len(gene_bins)
    selected = len(transformer.selected_bins_)

    print(
        f"bin_size={bin_size:3d} | "
        f"전체 후보={total:,} | "
        f"선택={selected:,} | "
        f"선택 비율={selected / total:.2%}"
    )

bin_size= 25 | 전체 후보=85,500 | 선택=13,446 | 선택 비율=15.73%
bin_size= 50 | 전체 후보=49,116 | 선택=23,329 | 선택 비율=47.50%
bin_size=100 | 전체 후보=26,268 | 선택=21,844 | 선택 비율=83.16%


⬆️ 해석
- 25: 구간이 세분화되어 후보가 많지만, 5명 이상에서 발견되는 구간은 적음.
- 50: 적절히 묶인 구간이 늘면서 선택된 피처 수가 가장 많음.
- 100: 구간이 넓어지면서 대부분의 후보가 5명 이상 조건을 충족함.

In [7]:
from scipy.sparse import csr_matrix, hstack
from sklearn.base import BaseEstimator, TransformerMixin

# 피처 구성 E
def make_features_E(X):
    X_binary = binarize_mutations(X)
    X_count = np.log1p(mutation_count(X))
    X_types = np.log1p(mutation_type_counts(X))
    X_pairs = np.log1p(amino_acid_pair_counts(X))
    X_trunc = gene_truncation_flags(X)

    return pd.concat(
        [X_binary, X_count, X_types, X_pairs, X_trunc],
        axis=1
    )


def make_features_E_sparse(X):
    return csr_matrix(
        make_features_E(X).to_numpy(dtype=np.float32)
    )


# 유전자별 Hotspot 위치
HOTSPOTS = {
    "IDH1": ["R132"],
    "IDH2": ["R140", "R172"],
    "BRAF": ["V600"],
    "HRAS": ["Q61", "G13", "G12"],
    "KIT": ["D816", "N822", "K642"],
    "CTNNB1": ["S33", "S37", "S45", "T41", "G34", "D32"],
    "U2AF1": ["S34"],
    "PIK3CA": ["E545", "H1047", "E542"],
    "EGFR": ["L858", "G598", "A289"],
    "RET": ["M918"],
    "RAC1": ["P29"],
    "SPOP": ["F133", "W131", "F102", "Y87"],
    "EZH2": ["Y641", "Y646"],
    "TP53": ["R175", "R248", "R273"],
    "PTEN": ["R130"],
}

assert sum(len(v) for v in HOTSPOTS.values()) == 35


def make_hotspot_features(X):
    hotspot_data = {}

    for gene, positions in HOTSPOTS.items():
        if gene not in X.columns:
            raise ValueError(f"유전자 컬럼 없음: {gene}")

        tokens = X[gene].fillna("WT").astype(str).str.split()

        for position in positions:
            col_name = f"hotspot_{gene}_{position}"

            hotspot_data[col_name] = tokens.map(
                lambda variants: int(
                    any(v.startswith(position) for v in variants)
                )
            ).astype(np.int8)

    if "NPM1" not in X.columns:
        raise ValueError("유전자 컬럼 없음: NPM1")

    hotspot_data["hotspot_NPM1_fs"] = (
        X["NPM1"]
        .fillna("WT")
        .astype(str)
        .str.contains("fs", case=False, regex=False)
        .astype(np.int8)
    )

    return pd.DataFrame(hotspot_data, index=X.index)


class FeaturesEWithHotspotAndPositionBins(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )
        self.position_transformer_.fit(X)
        return self

    def transform(self, X):
        X_base = make_features_E_sparse(X)

        X_hotspot = csr_matrix(
            make_hotspot_features(X).to_numpy(dtype=np.float32)
        )

        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_hotspot, X_position],
            format="csr",
            dtype=np.float32
        )


class FeaturesEWithHotspot3AndPositionBins(
    FeaturesEWithHotspotAndPositionBins
):
    def transform(self, X):
        X_base = make_features_E_sparse(X)

        X_hotspot = csr_matrix(
            make_hotspot_features(X).to_numpy(dtype=np.float32) * 3
        )

        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_hotspot, X_position],
            format="csr",
            dtype=np.float32
        )

In [8]:
print("Hotspot 개수:", sum(len(v) for v in HOTSPOTS.values()) + 1)

for bin_size in [25, 50, 100]:
    model_features = FeaturesEWithHotspot3AndPositionBins(
        bin_size=bin_size,
        min_patients=5
    )
    print(f"bin_size={bin_size}: 생성 가능")

Hotspot 개수: 36
bin_size=25: 생성 가능
bin_size=50: 생성 가능
bin_size=100: 생성 가능


## 2. CV 실험

### 2-1. 기준 모델(bin_size=50) 재현

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 기존 최고 성능 모델 재현
pipeline_bin50 = Pipeline([
    ("features", FeaturesEWithHotspot3AndPositionBins(
        bin_size=50,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

cv_bin50 = run_cv(
    pipeline_bin50,
    X,
    y,
    folds=folds
)

print("\n===== 기준 모델 재현 결과 =====")
print(f"CV Macro F1: {cv_bin50['mean']:.5f}")
print(f"기존 기록: 0.5085")

fold 1: Macro F1 = 0.5142
fold 2: Macro F1 = 0.5117
fold 3: Macro F1 = 0.4952
fold 4: Macro F1 = 0.5210
fold 5: Macro F1 = 0.5003
CV Macro F1 = 0.5085 (std 0.0094)

===== 기준 모델 재현 결과 =====
CV Macro F1: 0.50846
기존 기록: 0.5085


### 2-2. bin_size=25, 100 실험

In [10]:
# 기존 bin_size=50 결과 보관
bin_results = {50: cv_bin50}

# 새로운 bin 크기 실험
for bin_size in [25, 100]:
    print(f"\n{'=' * 50}")
    print(f"Position-bin 크기: {bin_size}")
    print(f"{'=' * 50}")

    pipeline = Pipeline([
        ("features", FeaturesEWithHotspot3AndPositionBins(
            bin_size=bin_size,
            min_patients=5
        )),
        ("model", LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=3000,
            solver="lbfgs"
        ))
    ])

    cv_result = run_cv(
        pipeline,
        X,
        y,
        folds=folds
    )

    bin_results[bin_size] = cv_result


Position-bin 크기: 25
fold 1: Macro F1 = 0.5104
fold 2: Macro F1 = 0.5251
fold 3: Macro F1 = 0.5005
fold 4: Macro F1 = 0.5254
fold 5: Macro F1 = 0.5008
CV Macro F1 = 0.5125 (std 0.0111)

Position-bin 크기: 100
fold 1: Macro F1 = 0.5152
fold 2: Macro F1 = 0.5093
fold 3: Macro F1 = 0.4878
fold 4: Macro F1 = 0.5200
fold 5: Macro F1 = 0.5022
CV Macro F1 = 0.5069 (std 0.0113)


In [11]:
results_df = pd.DataFrame([
    {
        "bin_size": size,
        "cv_macro_f1": result["mean"],
        "diff_vs_50": result["mean"] - bin_results[50]["mean"]
    }
    for size, result in bin_results.items()
]).sort_values("cv_macro_f1", ascending=False)

display(results_df)

,bin_size,cv_macro_f1,diff_vs_50
1,25,0.512463,0.003999
0,50,0.508463,0.000000
2,100,0.506889,-0.001574


## 3. bin_size=25 홀드아웃

In [12]:
# 홀드아웃 검증 후보: bin_size=25

pipeline_bin25 = Pipeline([
    ("features", FeaturesEWithHotspot3AndPositionBins(
        bin_size=25,
        min_patients=5
    )),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

print("홀드아웃 검증 후보 모델")
print(pipeline_bin25)
print(f"개발 CV Macro F1: {bin_results[25]['mean']:.5f}")

홀드아웃 검증 후보 모델
Pipeline(steps=[('features', FeaturesEWithHotspot3AndPositionBins(bin_size=25)),
                ('model',
                 LogisticRegression(C=0.1, class_weight='balanced',
                                    max_iter=3000))])
개발 CV Macro F1: 0.51246


In [15]:
import importlib.util

manifest_path = PROJECT_ROOT / "scripts" / "fixed_holdout_manifest.py"

spec = importlib.util.spec_from_file_location(
    "fixed_holdout_manifest_updated",
    manifest_path
)

holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)

dev_idx, hold_idx = holdout_module.verify()

print("개발 데이터:", len(dev_idx))
print("홀드아웃 데이터:", len(hold_idx))
print("중복 인덱스:", len(set(dev_idx) & set(hold_idx)))

VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
개발 데이터: 4960
홀드아웃 데이터: 1241
중복 인덱스: 0


In [16]:
import importlib.util
from sklearn.base import clone

# 팀 공통 홀드아웃 분할 스크립트
manifest_path = PROJECT_ROOT / "scripts" / "fixed_holdout_manifest.py"

if not manifest_path.is_file():
    raise FileNotFoundError(manifest_path)

spec = importlib.util.spec_from_file_location(
    "fixed_holdout_manifest",
    manifest_path
)

holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)

# 공식 개발 / 홀드아웃 인덱스
dev_idx, hold_idx = holdout_module.verify()

assert len(dev_idx) == 4960
assert len(hold_idx) == 1241
assert len(set(dev_idx) & set(hold_idx)) == 0

# 개발 데이터만 사용
X_dev = X.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]

# 팀 공통 CV 분할
dev_folds = get_folds(
    y_dev,
    X=X_dev,
    classes=le.classes_
)

print("개발 데이터:", X_dev.shape)
print("홀드아웃 데이터 수:", len(hold_idx))
print("개발 Fold 수:", len(dev_folds))

# bin_size=25 vs 50
dev_results = {}

for bin_size in [50, 25]:
    print(f"\n{'=' * 45}")
    print(f"공식 개발 CV | bin_size={bin_size}")
    print(f"{'=' * 45}")

    pipeline = Pipeline([
        ("features", FeaturesEWithHotspot3AndPositionBins(
            bin_size=bin_size,
            min_patients=5
        )),
        ("model", LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=3000,
            solver="lbfgs"
        ))
    ])

    result = run_cv(
        pipeline,
        X_dev,
        y_dev,
        folds=dev_folds
    )

    dev_results[bin_size] = result

print("\n===== 공식 개발 CV 비교 =====")
print(f"bin_size=50: {dev_results[50]['mean']:.5f}")
print(f"bin_size=25: {dev_results[25]['mean']:.5f}")

delta = dev_results[25]["mean"] - dev_results[50]["mean"]
print(f"성능 차이: {delta:+.5f}")

if delta > 0:
    print("개발 CV 개선 → 홀드아웃 검증 후보")
else:
    print("개발 CV 미개선 → 홀드아웃 검증 보류")
    

VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
개발 데이터: (4960, 4384)
홀드아웃 데이터 수: 1241
개발 Fold 수: 5

공식 개발 CV | bin_size=50
fold 1: Macro F1 = 0.5019
fold 2: Macro F1 = 0.4762
fold 3: Macro F1 = 0.4605
fold 4: Macro F1 = 0.4881
fold 5: Macro F1 = 0.5047
CV Macro F1 = 0.4863 (std 0.0165)

공식 개발 CV | bin_size=25
fold 1: Macro F1 = 0.4993
fold 2: Macro F1 = 0.4866
fold 3: Macro F1 = 0.4816
fold 4: Macro F1 = 0.4880
fold 5: Macro F1 = 0.5080
CV Macro F1 = 0.4927 (std 0.0096)

===== 공식 개발 CV 비교 =====
bin_size=50: 0.48628
bin_size=25: 0.49270
성능 차이: +0.00642
개발 CV 개선 → 홀드아웃 검증 후보


In [17]:
# 공식 홀드아웃 성능 비교

from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 공식 홀드아웃 데이터
X_hold = X.iloc[hold_idx].reset_index(drop=True)
y_hold = np.asarray(y)[hold_idx]

# 기존 개발 데이터 재사용
assert len(X_dev) == 4960
assert len(X_hold) == 1241

holdout_results = {}

for bin_size in [50, 25]:
    print(f"\n{'=' * 45}")
    print(f"홀드아웃 검증 | bin_size={bin_size}")
    print(f"{'=' * 45}")

    pipeline = Pipeline([
        ("features", FeaturesEWithHotspot3AndPositionBins(
            bin_size=bin_size,
            min_patients=5
        )),
        ("model", LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=3000,
            solver="lbfgs"
        ))
    ])

    # 개발 데이터로만 학습
    pipeline.fit(X_dev, y_dev)

    # 홀드아웃 예측
    y_pred = pipeline.predict(X_hold)

    score = f1_score(
        y_hold,
        y_pred,
        average="macro",
        zero_division=0
    )

    holdout_results[bin_size] = score
    print(f"Holdout Macro F1: {score:.5f}")

print("\n===== 홀드아웃 비교 =====")
print(f"bin_size=50: {holdout_results[50]:.5f}")
print(f"bin_size=25: {holdout_results[25]:.5f}")

delta = holdout_results[25] - holdout_results[50]
print(f"성능 차이: {delta:+.5f}")

if delta > 0:
    print("홀드아웃 개선 확인")
else:
    print("홀드아웃 개선 없음")


홀드아웃 검증 | bin_size=50
Holdout Macro F1: 0.50570

홀드아웃 검증 | bin_size=25
Holdout Macro F1: 0.50723

===== 홀드아웃 비교 =====
bin_size=50: 0.50570
bin_size=25: 0.50723
성능 차이: +0.00153
홀드아웃 개선 확인


1. Position-bin 25의 개선 효과가 홀드아웃에서도 확인됨

   개발 CV에서 +0.00642, 홀드아웃에서 +0.00153 개선됨. 다만 홀드아웃 개선 폭은 작기 때문에 확실한 일반화 성능 향상이라고 단정하기는 어려움.


2. 더 세밀한 위치 정보가 도움이 될 가능성

   50칸보다 25칸으로 구간을 나눴을 때 성능이 높아짐. 암종 분류에 세밀한 변이 위치 정보가 유용할 가능성을 보여주지만, 원인을 확정한 건 아님.

## 4. 실험 로그 저장

In [18]:
import pandas as pd
from pathlib import Path

log_path = PROJECT_ROOT / "experiments" / "position_bin_tuning_results.csv"
log_path.parent.mkdir(parents=True, exist_ok=True)

results_df = pd.DataFrame([
    {
        "experiment": "LR_E_hotspot_x3_position_bin",
        "bin_size": bin_size,
        "min_patients": 5,
        "C": 0.1,
        "class_weight": "balanced",
        "dev_cv_macro_f1": dev_results[bin_size]["mean"],
        "holdout_macro_f1": holdout_results[bin_size],
    }
    for bin_size in [50, 25]
])

results_df.to_csv(log_path, index=False)

display(results_df)
print(f"저장 완료: {log_path}")

,experiment,bin_size,min_patients,C,class_weight,dev_cv_macro_f1,holdout_macro_f1
0,LR_E_hotspot_x3_position_bin,50,5,0.1,balanced,0.486284,0.505698
1,LR_E_hotspot_x3_position_bin,25,5,0.1,balanced,0.492701,0.507227


저장 완료: /Users/admin/Documents/ah07-cancer-subtype/experiments/position_bin_tuning_results.csv


# E + Position-bin 관련 실험

## 1. 실험 관련 준비

In [20]:
class FeaturesEWithPositionBins(
    FeaturesEWithHotspotAndPositionBins
):
    def transform(self, X):
        # 기본 피처 E
        X_base = make_features_E_sparse(X)

        # Position-bin 피처
        X_position = self.position_transformer_.transform(X)

        # Hotspot 없이 E + Position-bin만 결합
        return hstack(
            [X_base, X_position],
            format="csr",
            dtype=np.float32
        )

## 2. CV 실험

In [22]:
# 공식 개발 CV | LR(E + Position-bin)
# bin_size=50 vs 25

import time

dev_results_no_hotspot = {}

for bin_size in [50, 25]:
    print(f"\n{'=' * 45}")
    print(f"공식 개발 CV | Hotspot 제외 | bin_size={bin_size}")
    print(f"{'=' * 45}")

    pipeline = Pipeline([
        ("features", FeaturesEWithPositionBins(
            bin_size=bin_size,
            min_patients=5
        )),
        ("model", LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=3000,
            solver="lbfgs"
        ))
    ])

    start = time.perf_counter()

    result = run_cv(
        pipeline,
        X_dev,
        y_dev,
        folds=dev_folds
    )

    elapsed = time.perf_counter() - start

    dev_results_no_hotspot[bin_size] = result

    print(f"실행 시간: {elapsed:.1f}초")

print("\n===== 공식 개발 CV 비교 (Hotspot 제외) =====")

score_50 = dev_results_no_hotspot[50]["mean"]
score_25 = dev_results_no_hotspot[25]["mean"]

print(f"bin_size=50: {score_50:.5f}")
print(f"bin_size=25: {score_25:.5f}")
print(f"성능 차이: {score_25 - score_50:+.5f}")


공식 개발 CV | Hotspot 제외 | bin_size=50
fold 1: Macro F1 = 0.5133
fold 2: Macro F1 = 0.4629
fold 3: Macro F1 = 0.4629
fold 4: Macro F1 = 0.4909
fold 5: Macro F1 = 0.4988
CV Macro F1 = 0.4858 (std 0.0200)
실행 시간: 285.9초

공식 개발 CV | Hotspot 제외 | bin_size=25
fold 1: Macro F1 = 0.5096
fold 2: Macro F1 = 0.4680
fold 3: Macro F1 = 0.4878
fold 4: Macro F1 = 0.4854
fold 5: Macro F1 = 0.4995
CV Macro F1 = 0.4901 (std 0.0140)
실행 시간: 279.8초

===== 공식 개발 CV 비교 (Hotspot 제외) =====
bin_size=50: 0.48579
bin_size=25: 0.49005
성능 차이: +0.00427


## 3. bin_size=25 홀드아웃

In [23]:
holdout_results_no_hotspot = {}

for bin_size in [50, 25]:
    print(f"\n{'=' * 45}")
    print(f"공식 홀드아웃 | Hotspot 제외 | bin_size={bin_size}")
    print(f"{'=' * 45}")

    pipeline = Pipeline([
        ("features", FeaturesEWithPositionBins(
            bin_size=bin_size,
            min_patients=5
        )),
        ("model", LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=3000,
            solver="lbfgs"
        ))
    ])

    start = time.perf_counter()

    pipeline.fit(X_dev, y_dev)
    y_pred = pipeline.predict(X_hold)

    score = f1_score(
        y_hold,
        y_pred,
        average="macro",
        zero_division=0
    )

    elapsed = time.perf_counter() - start

    holdout_results_no_hotspot[bin_size] = score

    print(f"Holdout Macro F1: {score:.5f}")
    print(f"실행 시간: {elapsed:.1f}초")

print("\n===== 공식 홀드아웃 비교 (Hotspot 제외) =====")
print(f"bin_size=50: {holdout_results_no_hotspot[50]:.5f}")
print(f"bin_size=25: {holdout_results_no_hotspot[25]:.5f}")

delta = (
    holdout_results_no_hotspot[25]
    - holdout_results_no_hotspot[50]
)

print(f"성능 차이: {delta:+.5f}")


공식 홀드아웃 | Hotspot 제외 | bin_size=50
Holdout Macro F1: 0.49908
실행 시간: 72.2초

공식 홀드아웃 | Hotspot 제외 | bin_size=25
Holdout Macro F1: 0.49686
실행 시간: 69.9초

===== 공식 홀드아웃 비교 (Hotspot 제외) =====
bin_size=50: 0.49908
bin_size=25: 0.49686
성능 차이: -0.00222


## 4. 실험 로그 저장

In [24]:
import pandas as pd

log_path = PROJECT_ROOT / "experiments" / "position_bin_tuning_results.csv"

# 기존 실험 로그 불러오기
log_df = pd.read_csv(log_path)

# 이번 실험 결과
new_results = pd.DataFrame([
    {
        "experiment": "LR_E_position_bin",
        "bin_size": bin_size,
        "min_patients": 5,
        "C": 0.1,
        "class_weight": "balanced",
        "dev_cv_macro_f1": dev_results_no_hotspot[bin_size]["mean"],
        "holdout_macro_f1": holdout_results_no_hotspot[bin_size],
    }
    for bin_size in [50, 25]
])

# 기존 동일 실험 기록이 있다면 제거 (중복 방지)
log_df = log_df[
    ~(
        (log_df["experiment"] == "LR_E_position_bin")
        & (log_df["bin_size"].isin([50, 25]))
    )
]

# 기존 기록 + 신규 기록
log_df = pd.concat(
    [log_df, new_results],
    ignore_index=True
)

log_df.to_csv(log_path, index=False)

display(log_df)
print(f"저장 완료: {log_path}")

,experiment,bin_size,min_patients,C,class_weight,dev_cv_macro_f1,holdout_macro_f1
0,LR_E_hotspot_x3_position_bin,50,5,0.1,balanced,0.486284,0.505698
1,LR_E_hotspot_x3_position_bin,25,5,0.1,balanced,0.492701,0.507227
2,LR_E_position_bin,50,5,0.1,balanced,0.485788,0.499084
3,LR_E_position_bin,25,5,0.1,balanced,0.490054,0.496864


저장 완료: /Users/admin/Documents/ah07-cancer-subtype/experiments/position_bin_tuning_results.csv


# OOF·test 예측 저장

In [25]:
# 기존 CV 결과 확인

print("bin_size=25:", dev_results_no_hotspot[25].keys())
print("bin_size=50:", dev_results_no_hotspot[50].keys())

bin_size=25: dict_keys(['scores', 'mean', 'std', 'oof', 'oof_proba'])
bin_size=50: dict_keys(['scores', 'mean', 'std', 'oof', 'oof_proba'])


In [26]:
# 핫스팟 제외 OOF 저장

import numpy as np

pred_dir = PROJECT_ROOT / "experiments" / "preds"
pred_dir.mkdir(parents=True, exist_ok=True)

for bin_size in [50, 25]:
    oof_proba = np.asarray(
        dev_results_no_hotspot[bin_size]["oof_proba"]
    )

    print(f"bin_size={bin_size}")
    print("OOF shape:", oof_proba.shape)

    assert oof_proba.shape == (4960, len(le.classes_))
    assert np.isfinite(oof_proba).all()
    assert np.allclose(oof_proba.sum(axis=1), 1, atol=1e-5)

    save_path = (
        pred_dir / f"H_data__E_position_bin_{bin_size}__dev_oof.npy"
    )

    np.save(save_path, oof_proba)
    print("저장 완료:", save_path)
    print()

bin_size=50
OOF shape: (4960, 26)
저장 완료: /Users/admin/Documents/ah07-cancer-subtype/experiments/preds/H_data__E_position_bin_50__dev_oof.npy

bin_size=25
OOF shape: (4960, 26)
저장 완료: /Users/admin/Documents/ah07-cancer-subtype/experiments/preds/H_data__E_position_bin_25__dev_oof.npy



In [27]:
# 핫스팟 포함 OOF 저장

for bin_size in [50, 25]:
    oof_proba = np.asarray(
        dev_results[bin_size]["oof_proba"]
    )

    assert oof_proba.shape == (4960, len(le.classes_))
    assert np.isfinite(oof_proba).all()
    assert np.allclose(oof_proba.sum(axis=1), 1, atol=1e-5)

    save_path = (
        pred_dir /
        f"H_data__E_hotspot_x3_position_bin_{bin_size}__dev_oof.npy"
    )

    np.save(save_path, oof_proba)

    print(f"bin_size={bin_size}")
    print("OOF shape:", oof_proba.shape)
    print("저장 완료:", save_path)
    print()

bin_size=50
OOF shape: (4960, 26)
저장 완료: /Users/admin/Documents/ah07-cancer-subtype/experiments/preds/H_data__E_hotspot_x3_position_bin_50__dev_oof.npy

bin_size=25
OOF shape: (4960, 26)
저장 완료: /Users/admin/Documents/ah07-cancer-subtype/experiments/preds/H_data__E_hotspot_x3_position_bin_25__dev_oof.npy



# 후속 실험 계획 및 보류 사항

- **완료:** Position-bin 크기(25, 50) 및 Hotspot×3 포함 여부에 따른 5-Fold CV와 Holdout 성능 비교
- **완료:** 4개 실험 조합의 개발 데이터 OOF 예측 확률 생성 및 저장
- **완료:** 실험 성능 비교 결과 CSV 저장
- **보류:** 앙상블 적용을 위한 Test 예측 생성 및 저장
- **보류 사유:** 앙상블 성능 개선 여부를 확인한 후 진행하기로 결정하여 현재 미실시

※ 본 노트북은 Train 데이터 기반 피처 비교 및 검증 실험을 목적으로 하며, Test 예측은 수행하지 않음.